# 04 — K-Means Clustering + Elbow / Silhouette / Davies-Bouldin (FINAL)

**Goal (paper §III):** 4-scenario comparison (Standard vs Robust) x (Point vs Window-60s), Elbow k=2..8, final centroids mapped to Coley (2019) regimes: Active Drilling / Circulation / Connection-Tripping / Idle.
Run heavy here (Kaggle, CPU). EDA notebooks already validated sampling.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score

SENSOR_8=['Rate Of Penetration (ft_per_hr)','Weight on Bit (klbs)','Rotary RPM (RPM)','Standpipe Pressure (psi)','Rotary Torque (kft_lb)','Hook Load (klbs)','Total Pump Output (gal_per_min)','Differential Pressure (psi)']
CSV_NAME='56-32 1sec data 27029986.csv'
N, K_RANGE, SEED = 120000, range(2,9), 42

def resolve_input(csv=CSV_NAME):
    root = Path('/kaggle/input')
    if root.exists():
        hits = list(root.rglob(csv))
        if hits:
            return hits[0]
        for p in root.rglob('*.csv'):
            if '56-32' in p.name or '1sec' in p.name or '27029986' in p.name:
                return p
        csvs = [p for p in root.rglob('*.csv') if p.is_file()]
        if csvs:
            return max(csvs, key=lambda p: p.stat().st_size)
        pqs = [p for p in root.rglob('*.parquet') if p.is_file()]
        if pqs:
            return max(pqs, key=lambda p: p.stat().st_size)
    for c in [Path('dataset')/csv, Path('../dataset')/csv, Path.cwd()/'dataset'/csv]:
        if c.exists(): return c
    listing = [str(p) for p in sorted(root.rglob('*'))][:30] if root.exists() else ['</kaggle/input not found>']
    raise FileNotFoundError('Dataset CSV not found. /kaggle/input contains: ' + str(listing) + ' | Fix: right panel Add Input -> add your private dataset -> re-run.')
if Path('/kaggle/working').exists(): OUT = Path('/kaggle/working')  # live session: visible in file browser, downloadable, saved on Save Version
elif Path('/kaggle/output').exists(): OUT = Path('/kaggle/output')
else: OUT = Path('../figures')
OUT.mkdir(parents=True, exist_ok=True)
print(resolve_input())

In [ ]:
usecols=['YYYY/MM/DD','HH:MM:SS']+SENSOR_8
# Evenly-spaced sample of N rows from head segment (fast, RAM-safe). For full 2.5M run, loop chunks in Kaggle.
df=pd.read_csv(resolve_input(), usecols=usecols, nrows=N*2, low_memory=True)
rng=np.random.RandomState(SEED)
df=df.iloc[np.sort(rng.choice(len(df), N, replace=False))].reset_index(drop=True)
for c in SENSOR_8: df[c]=pd.to_numeric(df[c], errors='coerce').astype('float32')
df=df.replace(-999.25, np.nan).dropna().reset_index(drop=True)
df['datetime']=pd.to_datetime(df['YYYY/MM/DD']+' '+df['HH:MM:SS'], format='%Y/%m/%d %H:%M:%S', errors='coerce')
print(df.shape)

In [ ]:
def windowed(df, w=60, step=30):
    df=df.sort_values('datetime').reset_index(drop=True); rows=[]
    for s in range(0, len(df)-w+1, step):
        ch=df.iloc[s:s+w]; rec={'t0':ch['datetime'].iloc[0]}
        for c in SENSOR_8:
            v=ch[c].to_numpy(dtype='float64')
            rec[c+'__mean']=v.mean(); rec[c+'__std']=v.std(); rec[c+'__delta']=v[-1]-v[0]
        rows.append(rec)
    return pd.DataFrame(rows)

scenarios={'point-standard':(df[SENSOR_8].to_numpy(), StandardScaler()), 'point-robust':(df[SENSOR_8].to_numpy(), RobustScaler())}
win=windowed(df)
feat=[c for c in win.columns if c!='t0']
scenarios['window60-standard']=(win[feat].to_numpy(), StandardScaler())
scenarios['window60-robust']=(win[feat].to_numpy(), RobustScaler())
print({k: v[0].shape for k,v in scenarios.items()})

In [ ]:
rows=[]
for name,(X,sc) in scenarios.items():
    Xs=sc.fit_transform(X)
    # subsample 20k for metric speed
    idx=np.random.RandomState(SEED).choice(len(Xs), min(len(Xs),20000), replace=False)
    Xe=Xs[idx]
    for k in K_RANGE:
        lab=KMeans(n_clusters=k, n_init=10, random_state=SEED).fit_predict(Xe)
        rows.append({'scenario':name,'k':k,'inertia':float(KMeans(n_clusters=k,n_init=10,random_state=SEED).fit(Xe).inertia_), 'silhouette':float(silhouette_score(Xe,lab)), 'dbi':float(davies_bouldin_score(Xe,lab))})
res=pd.DataFrame(rows)
res.to_csv(OUT/'table4_elbow_metrics.csv', index=False)
display(res.pivot_table(index='k', columns='scenario', values='silhouette').round(4))

In [ ]:
fig, ax=plt.subplots(1,2, figsize=(12,4))
for name, g in res.groupby('scenario'):
    ax[0].plot(g['k'], g['inertia'], marker='o', label=name)
    ax[1].plot(g['k'], g['silhouette'], marker='o', label=name)
ax[0].set_title('Elbow — Inertia vs k'); ax[0].set_xlabel('k'); ax[1].set_title('Silhouette vs k (higher better)')
ax[1].set_xlabel('k'); ax[0].legend(fontsize=8); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig(OUT/'fig5_elbow_silhouette.png', dpi=150); plt.show()

In [ ]:
BEST_SCENARIO='window60-robust'; BEST_K=4  # adjust from elbow above
X, sc = scenarios[BEST_SCENARIO]
Xs=sc.fit_transform(X)
lab=KMeans(n_clusters=BEST_K, n_init=10, random_state=SEED).fit_predict(Xs)
cent=sc.inverse_transform(KMeans(n_clusters=BEST_K, n_init=10, random_state=SEED).fit(Xs).cluster_centers_)
if 'window' in BEST_SCENARIO:
    # collapse mean-features back to sensor units for physics reading
    meancols=[c for c in feat if c.endswith('__mean')]
    ctab=pd.DataFrame(cent[:, :len(meancols)], columns=[c.replace('__mean','') for c in meancols])
else:
    ctab=pd.DataFrame(cent, columns=SENSOR_8)
ctab.index.name='cluster'; display(ctab.round(2))
ctab.round(2).to_csv(OUT/'table5_centroids.csv')
pd.Series(lab).value_counts(normalize=True).round(4).to_csv(OUT/'table6_cluster_share.csv')
print('Physics hint: WOB>0+RPM>0+ROP>0 => Active Drilling; WOB~0+ROP=0+pump high => Circulation; pump~0+RPM~0 => Connection/Idle (Coley 2019).')

In [ ]:
p2=PCA(n_components=2, random_state=SEED).fit_transform(Xs[np.random.RandomState(SEED).choice(len(Xs), min(len(Xs),15000), replace=False)])
ls=lab[np.random.RandomState(SEED).choice(len(lab), p2.shape[0], replace=False)]
plt.figure(figsize=(6,5)); plt.scatter(p2[:,0], p2[:,1], c=ls, s=4, cmap='tab10', alpha=0.6)
plt.title(f'PCA 2D — {BEST_SCENARIO} k={BEST_K}')
plt.xlabel('PC1'); plt.ylabel('PC2'); plt.tight_layout()
plt.savefig(OUT/'fig6_pca_clusters.png', dpi=150); plt.show()

#### Cell: Export & verify — collect everything for download
**What:** lists what was actually written to `OUT`, copies it into `/kaggle/working` (the live, downloadable folder), and packs it into one `kaggle_outputs.zip`.
**Why:** if this cell reports `<EMPTY>`, the cells above never ran (error or skipped) — outputs cannot exist. The zip gives you one-click download instead of per-file clicks.
**Read this:** the printed file list (expect `table*.csv` + `fig*.png`) and the zip size; then download via the file browser.


In [ ]:
from pathlib import Path
import shutil, zipfile
OUT.mkdir(parents=True, exist_ok=True)
live = Path('/kaggle/working') if Path('/kaggle/working').exists() else OUT
moved = []
for f in sorted(OUT.glob('*')):
    if not f.is_file() or f.suffix == '.zip':
        continue
    t = live / f.name
    if f.resolve() != t.resolve():
        shutil.copy2(f, t)
    moved.append(f.name)
print('OUT =', OUT)
print('files:', moved if moved else '<EMPTY — cells above did not run or errored, re-run from the top>')
zp = live / 'kaggle_outputs.zip'
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for name in moved:
        z.write(live / name, name)
print(f'zip: {zp} ({zp.stat().st_size/1024:.1f} KB)')
print('Download: file browser ->', live, '-> kaggle_outputs.zip. For the paper link: Save Version -> Output tab.')

try:
    from IPython.display import FileLink, display as _display
    _display(FileLink(str(zp)))
    print('Click the link above to download the zip directly (no file browser needed).')
except Exception as _e:
    print('direct-link unavailable:', _e)
